# Categorical vs. Continuous Data

## Today

- What do we mean by *categorical* vs *continuous*?
- Why types matter for **summaries** and **visuals**


## What are some ways we can classify data?

- anecdotal vs. representative
- census vs. sample
- observational vs. experimental
- **categorical vs. numerical**
- **discrete vs. continuous**
- cross-sectional vs. time series
- longitudinal vs. panel
- unstructured vs. structured


## Variable Types

- Categorical
  - Binary - two categories
  - Nominal - multiple unordered categories
  - Ordinal - multiple ordered categories

- Numerical
  - Continuous - a range of numbers (measurement data)
  - Discrete - take on separate, distinct values (countable data)


## (Random) daily-life dataset

What are the variable types?


In [ ]:
library(tibble)
library(dplyr)
library(knitr) # package for pretty rendering of tables, best thing to use for .qmd

set.seed(7)

N <- 60

lifelog <- tibble(
  id = 1:N,
  age = sample(18:35, N, replace = TRUE),
  height_cm = round(rnorm(N, mean = 170, sd = 10), 1),
  commute_mode = sample(c("Walk","Bike","Transit","Car"), N, replace = TRUE),
  coffee_cups = sample(0:5, N, replace = TRUE),
  coffee_today = coffee_cups > 0,
  study_hours = round(runif(N, 0, 6), 1),
  mood = factor(
    sample(c("Low","Medium","High"), N, replace = TRUE),
    levels = c("Low","Medium","High"),
    ordered = TRUE
  ),
  zip_code = sample(c("20001","20002","20037","20052"), N, replace = TRUE)
)

kable(head(lifelog, 15)) # from knitr


## The Two Big Families

| Family | Meaning | R classes you'll see | Typical summaries | Typical visuals |
|---|---|---|---|---|
| **Categorical** | labels/groups (nominal or ordered) | `factor`, `ordered`, `character`, `logical` | counts, proportions | bar charts, stacked bars |
| **Continuous** | measurements in a range | `numeric`, `double`, `integer` | mean/median, sd/IQR, quantiles | histogram, line plot, scatterplot, boxplot |


# Categorical Data


## Classify

What counts as categorical?

- **Nominal**: `commute_mode`, `zip_code`
- **Ordinal**: `mood` (Low < Medium < High)
- **Binary**: `coffee_today` (TRUE/FALSE)


## Summarize one categorical variable


In [ ]:
lifelog_counts <- lifelog |>
  count(commute_mode) |>
  mutate(prop = n / sum(n)) |>
  arrange(desc(n))

lifelog_counts


## Visualize: single categorical variable (bar chart)


In [ ]:
library(ggplot2)

ggplot(lifelog, aes(x = commute_mode)) +
  geom_bar(fill="chartreuse4") +
  labs(x = "Commute mode", y = "Count")


## Visualize: single categorical variable (col chart)


In [ ]:
ggplot(lifelog_counts, aes(x = commute_mode, y = prop)) +
  geom_col(fill="chartreuse4") +
  labs(x = "Commute mode", y = "Proportion")


## Visualize: proportions with bar chart


In [ ]:
ggplot(lifelog, aes(x = commute_mode, y = after_stat(prop), group = 1)) +
  geom_bar(fill="chartreuse4") +
  labs(x = "Commute mode", y = "Proportion")


## Visualize: two categorical variables (dodged bars)


In [ ]:
ggplot(lifelog, aes(x = commute_mode, fill = mood)) +
  geom_bar(position = "dodge") +
  labs(x = "Commute mode", y = "Count", fill = "Mood (ordinal)")


## Visualize: two categorical variables (stacked proportions)


In [ ]:
library(scales)

ggplot(lifelog, aes(x = commute_mode, fill = mood)) +
  geom_bar(position = "fill") +
  scale_y_continuous(labels = percent) +
  labs(x = "Commute mode", y = "Proportion", fill = "Mood")


# Continuous Data


## Classify

What counts as continuous?

- **Measurements (conceptually continuous)**: `height_cm`, `study_hours`
- **Counts (discrete)**: `coffee_cups` (treated similarly but integer-valued)


## Summarize


In [ ]:
lifelog |>
  summarize(
    mean_height   = mean(height_cm),
    sd_height     = sd(height_cm),
    median_height = median(height_cm),
    q25_height    = quantile(height_cm, 0.25),
    q75_height    = quantile(height_cm, 0.75),
    mean_study    = mean(study_hours),
    sd_study      = sd(study_hours),
    median_study  = median(study_hours),
    q25_study     = quantile(study_hours, 0.25),
    q75_study     = quantile(study_hours, 0.75)
  )


## Visualize: one continuous variable (histogram)


In [ ]:
ggplot(lifelog, aes(x = height_cm)) +
  geom_histogram(binwidth = 5, fill="chartreuse4") +
  labs(x = "Height (cm)", y = "Count")


## Visualize: two continuous variables (scatterplot)


In [ ]:
ggplot(lifelog, aes(x = height_cm, y = study_hours)) +
  geom_point(color = "darkorange", alpha = 0.7) +
  labs(x = "Height (cm)", y = "Study hours (per day)")


## Visualize: continuous distributions across categories

Histogram by category (can be hard to read)


In [ ]:
ggplot(lifelog, aes(x = study_hours, fill = commute_mode)) +
  geom_histogram(binwidth = 0.5, alpha = 0.6, position = "identity", color = "white") +
  labs(
    x = "Study hours (per day)",
    y = "Count",
    fill = "Commute mode",
    title = "Distribution of study hours by commute mode"
  )


## Visualize: boxplot (continuous by categorical)


In [ ]:
ggplot(lifelog, aes(x = commute_mode, y = study_hours, fill = commute_mode)) +
  geom_boxplot() +
  labs(x = "Commute mode", y = "Study hours (per day)")

## Try it out!

**Classify: continuous or categorical (nominal, ordinal, binary)**  
- `coffee_today` → ?  
- `mood` → ?  
- `id`, `zip_code` → ?  

**Summarize:**  
- Categorical → ?  
- Continuous → ?

**Visualize**  
- `coffee_today` → ?  
- `coffee_cups` → ?  
- `id`, `zip_code` → ?


## Try it out! (answers)

**Classify:**  
- `coffee_today` → binary categorical  
- `mood` → ordinal categorical  
- `id`, `zip_code` → categorical identifiers  

**Summarize:**  
- Categorical → counts, proportions  
- Continuous → mean/median, SD/IQR, quantiles  

**Visualize:**  
- `coffee_today` → bar chart of counts/proportions (TRUE vs FALSE)  
- `coffee_cups` → histogram (or bar chart since discrete)  
- `id`, `zip_code` → usually don’t plot (zip_code counts if needed; id is just an identifier)


## Other practices (continuous)

- Don't take means of numerics used as labels like **IDs** or **ZIP codes**
- Check for **outliers**.


In [ ]:
lifelog |>
  summarize(q05 = quantile(height_cm, .05), q95 = quantile(height_cm, .95))


## Other practices (categorical)

- **Unseen levels**: set factor levels explicitly for consistent ordering


## Factor vs. vector of characters


In [ ]:
tbl <- tibble(month = c("March", "January", "February"))

# Default: character columns arrange alphabetically
tbl |> arrange(month)


In [ ]:
tbl <- tbl |>
  mutate(month = factor(month, levels = c("January", "February", "March")))

# Now arrange() follows factor levels
tbl |> arrange(month)


# Penguins raw

In [1]:
library(palmerpenguins)

penguins_cleaned <- penguins_raw |>
  select(
    -c(Comments,
       studyName,
       `Sample Number`,
       `Individual ID`,
       `Delta 15 N (o/oo)`,
       `Delta 13 C (o/oo)`)
  ) |> 
  filter(
    !is.na(Species),
    !is.na(Island),
    !is.na(`Culmen Length (mm)`),
    !is.na(`Culmen Depth (mm)`),
    !is.na(`Flipper Length (mm)`),
    !is.na(`Body Mass (g)`)
  ) |>
  mutate(
    # Binary (categorical)
    Sex = tolower(as.character(Sex)),

    #integers are discrete, but here we treat as numeric (continuous)
    `Body Mass (g)` = as.integer(`Body Mass (g)`),
    `Flipper Length (mm)` = as.integer(`Flipper Length (mm)`),

    # dates
    `Date Egg` = as.Date(`Date Egg`),

    # categorical variables
    Species = factor(Species),
    Region  = factor(Region),
    Island  = factor(Island),
    Stage   = factor(Stage),
    Sex     = factor(Sex),

    # logical - binary (categorical)
    `Clutch Completion` = ifelse(`Clutch Completion` == "Yes", TRUE, FALSE)
  )


Attaching package: ‘palmerpenguins’


The following objects are masked from ‘package:datasets’:

    penguins, penguins_raw




ERROR: Error in mutate(filter(select(penguins_raw, -c(Comments, studyName, `Sample Number`, : could not find function "mutate"
